# Milestone 2: Understanding & Reasoning Engine

## Task B2: Understanding & Reasoning Engine [20 Marks]

This notebook implements the core cognitive capabilities for understanding and reasoning with business intelligence data.

### Objectives:
- Implement sentiment analysis (Understanding)
- Build knowledge graphs for reasoning
- Train ML models for classification/prediction
- Integrate multiple cognitive capabilities


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
import sys
sys.path.append('../../')
from src.models.sentiment_analyzer import SentimentAnalyzer
from src.utils.text_preprocessor import TextPreprocessor

print("Libraries imported successfully!")


## Part 1: Understanding - Sentiment Analysis

Implement Natural Language Processing to understand customer sentiment from unstructured text.


In [ ]:
# Load processed data
try:
    df_reviews = pd.read_csv('../../data/processed/cleaned_reviews.csv')
    print(f"✅ Loaded {len(df_reviews)} processed reviews")
except FileNotFoundError:
    print("⚠️ Processed data not found. Please run Milestone 1 notebook first.")
    # Create sample data for demonstration
    df_reviews = pd.DataFrame({
        'text': ["Great service, very fast delivery", "Product was damaged", 
                 "Amazing quality, will buy again", "Late delivery but good product"] * 25,
        'cleaned_text': ["great service fast delivery", "product damaged",
                        "amazing quality buy again", "late delivery good product"] * 25
    })

df_reviews.head()


In [ ]:
# Initialize sentiment analyzer
analyzer = SentimentAnalyzer(method='vader')

# Analyze sentiment for all reviews
print("Analyzing sentiment for all reviews...")
sentiment_results = []

for text in df_reviews['cleaned_text']:
    if text and len(text) > 0:
        result = analyzer.analyze(text)
        sentiment_results.append(result)
    else:
        sentiment_results.append({'sentiment': 'neutral', 'compound': 0.0})

# Create sentiment dataframe
df_sentiment = pd.DataFrame(sentiment_results)
df_reviews_with_sentiment = pd.concat([df_reviews, df_sentiment], axis=1)

print("✅ Sentiment analysis completed!")
print(f"\nSentiment distribution:")
print(df_reviews_with_sentiment['sentiment'].value_counts())

df_reviews_with_sentiment.head()


In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment counts
sentiment_counts = df_reviews_with_sentiment['sentiment'].value_counts()
axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=['green', 'red', 'gray'])
axes[0].set_title('Sentiment Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(sentiment_counts.index, rotation=45)

# Compound score distribution
if 'compound' in df_reviews_with_sentiment.columns:
    axes[1].hist(df_reviews_with_sentiment['compound'], bins=30, edgecolor='black', color='skyblue')
    axes[1].set_title('Compound Sentiment Score Distribution')
    axes[1].set_xlabel('Compound Score')
    axes[1].set_ylabel('Frequency')
    axes[1].axvline(x=0, color='red', linestyle='--', label='Neutral')
    axes[1].legend()

plt.tight_layout()
plt.show()


## Part 2: Reasoning - Topic Modeling

Extract key topics from customer feedback to understand main themes and concerns.


In [ ]:
# Topic Modeling using LDA
from gensim import corpora, models
from gensim.models import LdaModel
from gensim.utils import simple_preprocess
from collections import Counter

# Prepare documents for topic modeling
documents = [text.split() for text in df_reviews_with_sentiment['cleaned_text'] if text and len(text) > 0]

# Create dictionary and corpus
dictionary = corpora.Dictionary(documents)
dictionary.filter_extremes(no_below=2, no_above=0.5)  # Filter rare and common words
corpus = [dictionary.doc2bow(doc) for doc in documents]

print(f"✅ Prepared {len(documents)} documents for topic modeling")
print(f"Dictionary size: {len(dictionary)} unique words")


In [ ]:
# Train LDA model
num_topics = 5  # Adjust based on your data
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

print("✅ LDA model trained successfully!")
print(f"\nTop topics identified:")
for idx, topic in lda_model.print_topics(-1, num_words=5):
    print(f"Topic {idx}: {topic}")


## Part 3: Reasoning - Knowledge Graph Construction

Build a knowledge graph to represent relationships between entities in customer feedback.


In [ ]:
# Knowledge Graph Construction
import networkx as nx
from collections import defaultdict

# Simple entity extraction (in production, use NER models)
# Extract common business-related entities
business_entities = {
    'service': ['service', 'delivery', 'support', 'customer service'],
    'product': ['product', 'quality', 'item', 'goods'],
    'price': ['price', 'cost', 'expensive', 'cheap', 'affordable'],
    'location': ['kampala', 'nakawa', 'kawempe', 'makindye'],
    'time': ['late', 'fast', 'quick', 'delay', 'on time']
}

# Build knowledge graph
G = nx.Graph()

# Add nodes (entities)
for entity_type, keywords in business_entities.items():
    G.add_node(entity_type, type='entity')

# Add sentiment nodes
G.add_node('positive', type='sentiment')
G.add_node('negative', type='sentiment')
G.add_node('neutral', type='sentiment')

# Extract relationships from reviews
for idx, row in df_reviews_with_sentiment.iterrows():
    text = str(row.get('cleaned_text', '')).lower()
    sentiment = row.get('sentiment', 'neutral')
    
    # Find entities mentioned in text
    mentioned_entities = []
    for entity_type, keywords in business_entities.items():
        if any(keyword in text for keyword in keywords):
            mentioned_entities.append(entity_type)
            G.add_node(entity_type, type='entity')
    
    # Create relationships: entity -> sentiment
    for entity in mentioned_entities:
        if not G.has_edge(entity, sentiment):
            G.add_edge(entity, sentiment, weight=1)
        else:
            G[entity][sentiment]['weight'] += 1

print(f"✅ Knowledge graph created!")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

# Visualize knowledge graph
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=1, iterations=50)
nx.draw(G, pos, with_labels=True, node_color='lightblue', 
        node_size=2000, font_size=10, font_weight='bold', 
        edge_color='gray', width=2, alpha=0.7)
plt.title("Knowledge Graph: Entity-Sentiment Relationships")
plt.show()


In [ ]:
# Save models and results
import pickle
import os

os.makedirs('../../data/models', exist_ok=True)
os.makedirs('../../data/processed', exist_ok=True)

# Save sentiment analyzer
with open('../../data/models/sentiment_analyzer.pkl', 'wb') as f:
    pickle.dump(analyzer, f)
print("✅ Sentiment analyzer saved")

# Save LDA model
lda_model.save('../../data/models/lda_model')
print("✅ LDA topic model saved")

# Save knowledge graph
with open('../../data/models/knowledge_graph.pkl', 'wb') as f:
    pickle.dump(G, f)
print("✅ Knowledge graph saved")

# Save results with sentiment
df_reviews_with_sentiment.to_csv('../../data/processed/reviews_with_sentiment.csv', index=False)
print("✅ Results saved")

print("\n=== Milestone 2 Summary ===")
print(f"✅ Sentiment analysis completed on {len(df_reviews_with_sentiment)} reviews")
print(f"✅ Topic modeling: {num_topics} topics identified")
print(f"✅ Knowledge graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


## Summary

This notebook has implemented the Understanding & Reasoning Engine:

1. ✅ **Understanding (NLP)**: Sentiment analysis to understand customer opinions
2. ✅ **Reasoning (Knowledge Graph)**: Entity-sentiment relationships
3. ✅ **Reasoning (Topic Modeling)**: Identified key topics in feedback
4. ✅ **Model Persistence**: Saved all models for use in interaction layer

### Cognitive Pillars Demonstrated:
- **Understand**: Sentiment analysis extracts meaning from unstructured text
- **Reason**: Knowledge graphs and topic models enable reasoning about customer feedback

### Next Steps:
- Enhance entity extraction with NER models
- Add predictive modeling for trend forecasting
- Integrate with interaction layer
